# Stage 1 - Data, knowledge base, retrieval and baselines
**DA624 - Improving Factuality of LLMs through RAG and Self-Verification**

This notebook re-implements Stage 1 on top of the shared `halluguard` package and **fixes several problems of the first version** (see `docs/CHANGES_FROM_STAGE1.md`):

| Problem in the first version | Fix |
|---|---|
| Knowledge base contained TruthfulQA's *correct and incorrect answer strings* (RAG could read the answer key) | KB = source documents only: HaluEval paragraphs (pooled) + Wikipedia pages cited by TruthfulQA |
| Failed API calls were stored as `"Error: ..."` answers and scored | failures raise; items are skipped and counted |
| `sleep(0.5)` = 120 req/min vs. a 30 req/min limit | token- and request-aware limiter, persistent cache, resumable runs |
| Recall@k / MRR / context coverage / prompt-vs-completion tokens promised but not computed | implemented |
| BM25 tokenisation left punctuation attached; zero-score docs entered the fusion | regex tokenizer + stop-words; zero scores dropped |

**Run order:** setup -> data -> knowledge base & retrieval -> baselines A/B -> evaluation. First time? use `PROFILE = "configs/quick.yaml"`.

In [ ]:
import os, sys
from pathlib import Path
import matplotlib; matplotlib.use("Agg")          # figures are written to results/ and shown with IPython.display.Image
from IPython.display import Image, display, Markdown
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.float_format", "{:.3f}".format)

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

# ---- choose how to run -----------------------------------------------------------------------------------------
PROFILE = None          # None = configs/default.yaml | "configs/quick.yaml" = small sanity run (recommended first)
MOCK = os.environ.get("HALLUGUARD_MOCK") == "1"      # offline smoke test with FAKE models - numbers are meaningless
from halluguard.config import load_config
cfg = load_config(PROFILE, mock=MOCK)
print("mock:", MOCK, "| generator:", cfg.llm.model, "| judge:", cfg.llm.judge_model, "| results ->", cfg.project.results_dir)

def show(name, stage):
    p = Path(cfg.project.results_dir) / stage / name
    display(Image(filename=str(p))) if p.exists() else print("(missing)", p)


## 1. Datasets and splits
HaluEval-QA (10,000 questions with gold paragraph, right and hallucinated answers) and TruthfulQA (790 questions, 475 cite Wikipedia). The three HaluEval pools (dev / test / end-to-end) are **disjoint** so detector calibration never sees the questions used for the final evaluation.

In [ ]:
from halluguard.experiments.common import load_resources
res = load_resources(cfg)                        # downloads data on first use, builds / loads the knowledge base
print({k: len(v) for k, v in res.splits.items()})
print("TruthfulQA questions with a Wikipedia source:", sum(bool(x["wiki_titles"]) for x in res.tqa), "/", len(res.tqa))
pd.DataFrame(res.splits["halueval_e2e"][:3])[["id", "question", "right_answer", "hallucinated_answer"]]

## 2. Knowledge base & hybrid retrieval
Dense (MiniLM + FAISS) and sparse (BM25) rankings are fused with Reciprocal Rank Fusion, `score(d) = sum 1/(k + rank)`, `k = 60`. Passage count and a retrieval sanity check:

In [ ]:
print("passages:", res.retriever.size)
q = res.splits["halueval_e2e"][0]
print("Q:", q["question"], "| gold answer:", q["right_answer"])
for h in res.retriever.search(q["question"], k=3):
    print(f"  [{h.rank}] rrf={h.score:.4f} dense_rank={h.dense_rank} bm25_rank={h.sparse_rank} | {h.text[:110]}...")

## 3. Baselines A (LLM only) and B (hybrid RAG) + retrieval evaluation
One call runs both baselines on the end-to-end pool, computes retrieval Recall@k / MRR (bm25 vs. dense vs. hybrid), context coverage, BERTScore, ROUGE-L, EM/F1, abstention rate, tokens and latency, and writes everything to `results/stage1/`.

In [ ]:
from halluguard.experiments import stage1
r1 = stage1.run(cfg, res)
r1["retrieval"]

### Retrieval quality
HaluEval has a known gold paragraph per question, so Recall@k and MRR are exact. TruthfulQA is evaluated at page level where a Wikipedia source exists.

In [ ]:
show("retrieval.png", "stage1"); r1["coverage"]

### Baseline comparison
`abstention_rate` matters: Baseline B is instructed to say when the context is insufficient, which is *not* a hallucination but also not an answer. Use the attempted-only columns and Stage 3's judge grades (which score abstentions separately) for a fair comparison.

In [ ]:
cols = ["dataset","system","n","abstention_rate","bertscore_f1","rougeL_f1","contains","tqa_truthful_sim","cost_calls","cost_prompt_tokens","cost_completion_tokens","cost_latency"]
r1["summary"][[c for c in cols if c in r1["summary"]]]

In [ ]:
show("quality.png", "stage1")

### Dataset EDA


In [ ]:
show("eda.png", "stage1")

## 4. What to look for / caveats
* `contains` (HaluEval) and `tqa_truthful_sim` (TruthfulQA) are judge-free correctness proxies; BERTScore / ROUGE against a 2-word entity answer are weak signals and are kept only for continuity with the original report.
* The KB is an *oracle-in-pool* setting: the gold paragraph is present among ~10k distractors. Retrieval is therefore easier than open-web RAG - report it as such.
* TruthfulQA questions without a Wikipedia source (`tqa_has_wiki = False`) have no gold evidence in the KB; RAG should abstain there, and Stage 3 reports them separately.
* Next: `02_hallucination_detection.ipynb`.